# Legal-BERT Hyperparameter Sensitivity Analysis
**Figure 8 — Validation F1 Heatmap (Learning Rate × Batch Size)**

Runs a 3×2 grid search for Legal-BERT fine-tuning on the COBS dataset and produces the heatmap chart for Section 5.5 of the thesis.

| Setting | Values |
|---------|--------|
| Learning Rate | 1e-5, 2e-5, 5e-5 |
| Batch Size | 8, 16 |
| Epochs | 3 (per run) |
| Max Length | 256 tokens |
| Base Model | `nlpaueb/legal-bert-base-uncased` |

**Setup:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
2. Upload `train.csv`, `val.csv`, `label_map.json` from `data/processed/` in Cell 3
3. Run all cells — total time ~20–30 min on T4 GPU

In [ ]:
# Cell 1 — Install only packages not pre-installed in Colab
# Colab already has: numpy, pandas, torch, matplotlib, seaborn, scikit-learn, datasets
# Do NOT reinstall them — that downgrades requests/numpy and breaks Colab's environment
!pip install -q transformers evaluate accelerate
print('Install complete. Dependency warnings above are non-fatal — safe to ignore.')

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Grid search will be very slow on CPU.')
    print('         Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Cell 3 — Upload data files
from google.colab import files
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/results',   exist_ok=True)

print('Upload these 3 files from your local  data/processed/  folder:')
print('  1. train.csv')
print('  2. val.csv')
print('  3. label_map.json')

uploaded = files.upload()

for fname in uploaded:
    dest = f'data/processed/{fname}'
    os.rename(fname, dest)
    print(f'  saved -> {dest}')

In [ ]:
# Cell 4 — Imports and constants
import pandas as pd
import numpy as np
import json, shutil, gc
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams['figure.dpi'] = 150

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
import evaluate

# ── Grid config ────────────────────────────────────────────────────────────
MODEL_NAME     = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN        = 256       # covers ~75% of COBS text; good speed/coverage balance
NUM_EPOCHS     = 3         # 3 epochs per run; sufficient to converge
SEED           = 42

LEARNING_RATES = [1e-5, 2e-5, 5e-5]
BATCH_SIZES    = [8, 16]

print(f'Grid: {len(LEARNING_RATES)} LRs x {len(BATCH_SIZES)} BSs = '
      f'{len(LEARNING_RATES)*len(BATCH_SIZES)} configurations')
print(f'Model : {MODEL_NAME}')
print(f'Tokens: max_length={MAX_LEN}, epochs={NUM_EPOCHS}')

In [ ]:
# Cell 5 — Load and inspect data
train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')

with open('data/processed/label_map.json') as f:
    lm = json.load(f)

ID2LABEL_FULL = {int(k): v for k, v in lm['id2label'].items()}

# Only labels present in train + val (E class has n=2 in val, D=0)
present_labels = sorted(
    int(x) for x in set(train_df['label'].unique()) | set(val_df['label'].unique())
)
NUM_LABELS = len(present_labels)
ID2LABEL   = {i: ID2LABEL_FULL[i] for i in present_labels}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}

print(f'Labels   : {ID2LABEL}')
print(f'Train    : {len(train_df)} samples')
print(f'Val      : {len(val_df)} samples')
print()
print('Train class distribution:')
print(train_df['label'].map(ID2LABEL).value_counts().sort_index())
print()
print('Val class distribution:')
print(val_df['label'].map(ID2LABEL).value_counts().sort_index())

In [ ]:
# Cell 6 — Tokenise (once, shared across all runs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,      # DataCollatorWithPadding pads per batch
    )

train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenise, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text',  'label']]).map(tokenise, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f'Train tokens — example: {len(train_ds[0]["input_ids"])} tokens')
print('Tokenisation complete.')

In [ ]:
# Cell 7 — Metric (macro-F1, consistent with original training)
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_metric.compute(
        predictions=preds,
        references=labels,
        average='macro',
    )['f1']
    return {'f1_macro': f1}

In [ ]:
# Cell 8 — Grid search (main loop)
# Estimated time on T4 GPU: ~3-5 min per run = 20-30 min total

results = []
total   = len(LEARNING_RATES) * len(BATCH_SIZES)
run_idx = 0

for lr in LEARNING_RATES:
    for bs in BATCH_SIZES:
        run_idx += 1
        run_tag  = f'lr{lr:.0e}_bs{bs}'
        out_dir  = f'tmp_grid/{run_tag}'

        print(f'[{run_idx}/{total}]  LR={lr:.0e}  BS={bs}  ({run_tag})')
        print('-' * 52)

        # Fresh model for every run — avoids weight leakage between configs
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
        )

        # Compute warmup_steps from ratio (warmup_ratio deprecated in v5.2)
        steps_per_epoch = -(-len(train_df) // bs)   # ceiling division
        total_steps     = steps_per_epoch * NUM_EPOCHS
        warmup_steps    = int(0.1 * total_steps)

        training_args = TrainingArguments(
            output_dir=out_dir,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=32,
            learning_rate=lr,
            warmup_steps=warmup_steps,
            weight_decay=0.01,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            seed=SEED,
            fp16=torch.cuda.is_available(),
            logging_steps=30,
            report_to='none',
            dataloader_num_workers=2,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,   # replaces 'tokenizer=' in transformers>=4.46
            data_collator=collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        # Best val macro-F1 across all epochs for this config
        best_f1 = max(
            log['eval_f1_macro']
            for log in trainer.state.log_history
            if 'eval_f1_macro' in log
        )
        best_f1_pct = round(best_f1 * 100, 2)

        results.append({'lr': lr, 'bs': bs, 'val_f1_macro': best_f1_pct})
        print(f'   => Best val macro-F1: {best_f1_pct:.2f}%')
        print()

        # Release GPU memory before next run
        del model, trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Remove checkpoints to save Colab disk space
        shutil.rmtree(out_dir, ignore_errors=True)

print('=' * 52)
print('Grid search complete!')
print()
print(f'{"LR":>10}  {"BS":>4}  {"Val Macro-F1 (%)"}' )
print('-' * 36)
for r in results:
    print(f'{r["lr"]:>10.0e}  {r["bs"]:>4}  {r["val_f1_macro"]:>17.2f}')

In [ ]:
# Cell 9 — Save results JSON
import json

os.makedirs('data/results', exist_ok=True)
with open('data/results/hyperparam_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved -> data/results/hyperparam_results.json')

In [ ]:
# Cell 10 — Build heatmap matrix
lr_vals = sorted(set(r['lr'] for r in results))
bs_vals = sorted(set(r['bs'] for r in results))

matrix = np.zeros((len(lr_vals), len(bs_vals)))
for r in results:
    i = lr_vals.index(r['lr'])
    j = bs_vals.index(r['bs'])
    matrix[i, j] = r['val_f1_macro']

lr_labels = [f'{lr:.0e}' for lr in lr_vals]   # '1e-05', '2e-05', '5e-05'
bs_labels = [f'BS = {bs}' for bs in bs_vals]

print('Heatmap matrix (rows=LR, cols=BS):')
print(pd.DataFrame(matrix, index=lr_labels, columns=bs_labels).round(2))

In [ ]:
# Cell 11 — Figure 8: Hyperparameter Sensitivity Heatmap
BG      = '#F5F3EF'
TEXT    = '#1E1E24'
BEST_C  = '#76BA70'   # green border on best cell

vmin = matrix.min() - 3
vmax = matrix.max() + 3

fig, ax = plt.subplots(figsize=(6.5, 5), dpi=150)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

im = ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=vmin, vmax=vmax, zorder=2)

best_val = matrix.max()
best_i, best_j = np.unravel_index(matrix.argmax(), matrix.shape)

for i in range(len(lr_vals)):
    for j in range(len(bs_vals)):
        val      = matrix[i, j]
        is_best  = (i == best_i and j == best_j)
        # switch text to white on dark cells, dark on light cells
        txt_col  = 'white' if val > (vmin + vmax) / 2 else TEXT
        ax.text(
            j, i, f'{val:.2f}%',
            ha='center', va='center', zorder=4,
            fontsize=13,
            fontweight='bold' if is_best else 'normal',
            color=txt_col,
        )
        if is_best:
            rect = plt.Rectangle(
                (j - 0.5, i - 0.5), 1, 1,
                linewidth=3, edgecolor=BEST_C,
                facecolor='none', zorder=5,
            )
            ax.add_patch(rect)

ax.set_xticks(range(len(bs_vals)))
ax.set_xticklabels(bs_labels, fontsize=11, color=TEXT)
ax.set_yticks(range(len(lr_vals)))
ax.set_yticklabels(lr_labels, fontsize=11, color=TEXT)
ax.set_xlabel('Batch Size', fontsize=12, color=TEXT, labelpad=8)
ax.set_ylabel('Learning Rate', fontsize=12, color=TEXT, labelpad=8)
ax.set_title(
    'Figure 8 — Legal-BERT Hyperparameter Sensitivity\n'
    'Validation Macro-F1 (%) across LR × Batch Size',
    fontsize=12, fontweight='bold', color=TEXT, pad=14,
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Macro-F1 (%)', fontsize=10, color=TEXT)
cbar.ax.yaxis.set_tick_params(labelcolor=TEXT)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False)

# Caption below figure
best_lr_str = lr_labels[best_i]
best_bs_val = bs_vals[best_j]
fig.text(
    0.5, 0.01,
    f'Best config: LR = {best_lr_str}, BS = {best_bs_val}  ->  F1 = {best_val:.2f}%   (green border)',
    ha='center', fontsize=8.5, color='#555555', style='italic',
)

plt.tight_layout(rect=[0, 0.04, 1, 1])

out_path = 'data/results/hyperparam_heatmap.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
print(f'Saved -> {out_path}')
plt.show()

In [ ]:
# Cell 12 — Print summary table for thesis Section 5.5
print('Table: Validation Macro-F1 across Hyperparameter Grid')
print('=' * 48)
header = f'{"Learning Rate":>15}  {"Batch Size":>10}  {"Val Macro-F1 (%)"}'
print(header)
print('-' * 48)
for r in sorted(results, key=lambda x: -x['val_f1_macro']):
    marker = ' <- BEST' if r['val_f1_macro'] == best_val else ''
    print(f'{r["lr"]:>15.0e}  {r["bs"]:>10}  {r["val_f1_macro"]:>17.2f}{marker}')
print()
print(f'Optimal: LR={lr_labels[best_i]}, BS={best_bs_val} -> F1={best_val:.2f}%')

In [ ]:
# Cell 13 — Download outputs to local machine
from google.colab import files

files.download('data/results/hyperparam_heatmap.png')
files.download('data/results/hyperparam_results.json')

print('Download complete.')
print('Copy hyperparam_results.json -> data/processed/ in your local workspace')
print('Copy hyperparam_heatmap.png  -> data/processed/ for the dashboard')